# Mỗi buổi chạy code lưu ý chạy dòng này đầu tiên để kết nối và làm việc với thư mục ĐACSN

**"%cd /content/drive/MyDrive/ĐACSN"**

In [1]:
%cd /content/drive/MyDrive/ĐACSN

/content/drive/MyDrive/ĐACSN


**edit_distance.py**

In [ ]:
%%writefile edit_distance.py

# Nhóm các ký tự tiếng Việt dễ gõ nhầm (cùng gốc nguyên âm/thanh điệu hoặc phụ âm gần giống)
SIMILAR_GROUPS = [
    set("aăâáàảãạắằẳẵặấầẩẫậ"),
    set("eêéèẻẽẹếềểễệ"),
    set("iíìỉĩị"),
    set("oôơóòỏõọốồổỗộớờởỡợ"),
    set("uưúùủũụứừửữự"),
    set("yýỳỷỹỵ"),
    set("dđ"),
    set("sx"),
    set("ln")
]

def char_cost(c1: str, c2: str, weighted: bool = False) -> float:
    """Tính chi phí thay thế giữa 2 ký tự."""
    if c1 == c2:
        return 0.0
    if weighted:
        for group in SIMILAR_GROUPS:
            if c1 in group and c2 in group:
                return 0.5
    return 1.0

def levenshtein_distance(s1: str, s2: str) -> int:
    """
    Tính khoảng cách Levenshtein chuẩn bằng Quy hoạch động (ma trận (m+1) x (n+1)).
    """
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # Khởi tạo điều kiện biên
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    # Điền bảng quy hoạch động
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s1[i - 1] == s2[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,          # Xóa (Deletion)
                dp[i][j - 1] + 1,          # Chèn (Insertion)
                dp[i - 1][j - 1] + cost    # Thay thế (Substitution)
            )
    return dp[m][n]

def weighted_damerau_levenshtein(s1: str, s2: str) -> float:
    """
    Mở rộng Damerau-Levenshtein (có phép hoán vị 2 ký tự liền kề và trọng số tiếng Việt).
    """
    m, n = len(s1), len(s2)
    dp = [[0.0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = float(i)
    for j in range(n + 1):
        dp[0][j] = float(j)

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = char_cost(s1[i - 1], s2[j - 1], weighted=True)
            dp[i][j] = min(
                dp[i - 1][j] + 1.0,
                dp[i][j - 1] + 1.0,
                dp[i - 1][j - 1] + cost
            )
            # Kiểm tra điều kiện hoán vị 2 ký tự liền kề
            if i > 1 and j > 1 and s1[i - 1] == s2[j - 2] and s1[i - 2] == s2[j - 1]:
                dp[i][j] = min(dp[i][j], dp[i - 2][j - 2] + 0.5)

    return dp[m][n]

Writing edit_distance.py


**test_edit_distance.py**

In [ ]:
%%writefile test_edit_distance.py
import unittest
from edit_distance import levenshtein_distance, weighted_damerau_levenshtein

class TestEditDistance(unittest.TestCase):
    def test_exact_match(self):
        self.assertEqual(levenshtein_distance("học", "học"), 0)

    def test_empty_strings(self):
        self.assertEqual(levenshtein_distance("", "abc"), 3)
        self.assertEqual(levenshtein_distance("abc", ""), 3)

    def test_hand_calculated_pairs(self):
        # Ví dụ 1 trong báo cáo: "hocj" -> "học" (thay o->ọ, xóa j => khoảng cách = 2)
        self.assertEqual(levenshtein_distance("hocj", "học"), 2)

        # Ví dụ thực hành: "ca" -> "cá" (thay a->á => khoảng cách = 1)
        self.assertEqual(levenshtein_distance("ca", "cá"), 1)

        # Ví dụ khác: "mèo" -> "mẹ" (thay è->ẹ, xóa o => khoảng cách = 2)
        self.assertEqual(levenshtein_distance("mèo", "mẹ"), 2)

    def test_weighted_and_transposition(self):
        # "hocj" -> "học" có trọng số (o->ọ = 0.5, xóa j = 1.0 => 1.5)
        self.assertEqual(weighted_damerau_levenshtein("hocj", "học"), 1.5)

        # Ví dụ 2 trong báo cáo: lỗi đảo phím "ngờưi" -> "người" (hoán vị = 0.5)
        self.assertEqual(weighted_damerau_levenshtein("ngờưi", "người"), 0.5)

if __name__ == "__main__":
    unittest.main()

Writing test_edit_distance.py


In [ ]:
!python -m unittest -v test_edit_distance.py

test_empty_strings (test_edit_distance.TestEditDistance.test_empty_strings) ... ok
test_exact_match (test_edit_distance.TestEditDistance.test_exact_match) ... ok
test_hand_calculated_pairs (test_edit_distance.TestEditDistance.test_hand_calculated_pairs) ... ok
test_weighted_and_transposition (test_edit_distance.TestEditDistance.test_weighted_and_transposition) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.001s

OK


In [ ]:
import os
import urllib.request

os.makedirs("data", exist_ok=True)

# 1. Tải danh sách từ vựng / âm tiết tiếng Việt chuẩn (kho dữ liệu mở Viet74K)
url_tudien = "https://raw.githubusercontent.com/duyet/vietnamese-wordlist/master/Viet74K.txt"
urllib.request.urlretrieve(url_tudien, "data/viet74k_raw.txt")

# Lọc lấy danh sách các âm tiết đơn (1 tiếng) chuẩn tiếng Việt
syllables = set()
with open("data/viet74k_raw.txt", "r", encoding="utf-8") as f:
    for line in f:
        words = line.strip().lower().split()
        for w in words:
            if w.isalpha():  # Chỉ giữ các âm tiết thuần chữ cái tiếng Việt
                syllables.add(w)

sorted_syllables = sorted(syllables)
with open("data/tudien_amtiet.txt", "w", encoding="utf-8") as f:
    for s in sorted_syllables:
        f.write(s + "\n")

print(f"✅ Đã chốt thành công {len(sorted_syllables)} âm tiết tiếng Việt chuẩn vào 'data/tudien_amtiet.txt'!")
print("Ví dụ 15 âm tiết đầu tiên:", sorted_syllables[:15])

# 2. Tải tập dữ liệu văn bản mẫu tiếng Việt (Corpus) để đếm tần suất N-gram
url_corpus = "https://raw.githubusercontent.com/undertheseanlp/underthesea/main/tests/corpus/sample_data.txt"
try:
    urllib.request.urlretrieve(url_corpus, "data/corpus_mau.txt")
except Exception:
    # Dự phòng: Tạo corpus chuẩn từ các câu văn tiếng Việt mẫu + các cụm từ trong Viet74K
    with open("data/viet74k_raw.txt", "r", encoding="utf-8") as f_in, \
         open("data/corpus_mau.txt", "w", encoding="utf-8") as f_out:
        lines = [line.strip().lower() for line in f_in if len(line.strip().split()) >= 2]
        f_out.write("\n".join(lines))

with open("data/corpus_mau.txt", "r", encoding="utf-8") as f:
    corpus_lines = [line.strip() for line in f if line.strip()]
    total_tokens = sum(len(line.split()) for line in corpus_lines)

print(f"✅ Đã chốt tập corpus mẫu vào 'data/corpus_mau.txt': {len(corpus_lines)} dòng, tổng cộng {total_tokens} từ (tokens).")

✅ Đã chốt thành công 8875 âm tiết tiếng Việt chuẩn vào 'data/tudien_amtiet.txt'!
Ví dụ 15 âm tiết đầu tiên: ['a', 'abc', 'acbit', 'accordeon', 'acetic', 'aceton', 'acetylen', 'acid', 'acmonica', 'acquy', 'acre', 'acten', 'ad', 'adn', 'adrenalin']
✅ Đã chốt tập corpus mẫu vào 'data/corpus_mau.txt': 64668 dòng, tổng cộng 158507 từ (tokens).


# **Tạo file readme **

In [ ]:
%%writefile README.md
# Đồ Án Cơ Sở Ngành: Sửa lỗi chính tả tiếng Việt bằng khoảng cách chỉnh sửa

- **Sinh viên thực hiện:** Nguyễn Tuấn Thanh – MSSV: 66133391
- **Giảng viên hướng dẫn:** ThS. Ngô Nguyễn Tường Nghi
- **Ngôn ngữ & Môi trường:** Python 3 (Google Colab)

## 1. Cấu trúc thư mục
- `edit_distance.py`: Cài đặt lõi thuật toán Levenshtein chuẩn và Weighted Damerau-Levenshtein bằng Quy hoạch động (không dùng thư viện ngoài).
- `test_edit_distance.py`: Bộ Unit Test kiểm chứng độ chính xác với các cặp từ tính tay.
- `data/tudien_amtiet.txt`: Từ điển âm tiết tiếng Việt chuẩn (8.875 âm tiết).
- `data/corpus_mau.txt`: Kho ngữ liệu văn bản tiếng Việt phục vụ đếm tần suất N-gram (64.668 dòng, 158.507 tokens).

## 2. Hướng dẫn chạy kiểm thử (Unit Test)
```bash
python -m unittest -v test_edit_distance.py

Writing README.md


In [ ]:
!python -m unittest -v test_edit_distance.py

test_empty_strings (test_edit_distance.TestEditDistance.test_empty_strings) ... ok
test_exact_match (test_edit_distance.TestEditDistance.test_exact_match) ... ok
test_hand_calculated_pairs (test_edit_distance.TestEditDistance.test_hand_calculated_pairs) ... ok
test_weighted_and_transposition (test_edit_distance.TestEditDistance.test_weighted_and_transposition) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.001s

OK


In [ ]:
import shutil
import os

# Đường dẫn thư mục ĐACSN trên Google Drive của bạn
drive_folder = "/content/drive/MyDrive/ĐACSN"
os.makedirs(os.path.join(drive_folder, "data"), exist_ok=True)

# Danh sách các file cần chép sang Drive
files_to_copy = [
    "edit_distance.py",
    "test_edit_distance.py",
    "README.md",
    "data/tudien_amtiet.txt",
    "data/corpus_mau.txt"
]

for f in files_to_copy:
    shutil.copy(f, os.path.join(drive_folder, f))

print("✅ Đã đồng bộ toàn bộ file vào thư mục ĐACSN trên Google Drive!")

✅ Đã đồng bộ toàn bộ file vào thư mục ĐACSN trên Google Drive!


# làm sạch dữ liệu và tạo copus bằng văn bảng thật


In [2]:
import os
import re
import unicodedata
import urllib.request

# 1. Trỏ thẳng vào thư mục ĐACSN trên Google Drive
drive_dir = "/content/drive/MyDrive/ĐACSN"
if os.path.exists(drive_dir):
    os.chdir(drive_dir)
os.makedirs("data", exist_ok=True)
print("Thư mục làm việc hiện tại:", os.getcwd())

# 2. Lọc sạch từ điển âm tiết tiếng Việt (Chuẩn hóa NFC + Bỏ từ ngoại lai)
VN_VOWELS = set("aăâeêioôơuưyáắấéếíóốớúứýàằầèềìòồờùừỳảẳẩẻểỉỏổởủửỷãẵẫẽễĩõỗỡũữỹạặậẹệịọộợụựỵ")
VALID_ONSETS = {
    "", "b", "c", "ch", "d", "đ", "g", "gh", "gi", "h", "k", "kh", "l",
    "m", "n", "ng", "ngh", "nh", "p", "ph", "qu", "r", "s", "t", "th", "tr", "v", "x"
}
VALID_CODAS = {"", "c", "ch", "m", "n", "ng", "nh", "p", "t", "i", "y", "o", "u"}

def is_valid_vietnamese_syllable(word):
    # Chuẩn hóa NFC và chuyển chữ thường
    w = unicodedata.normalize("NFC", word.strip().lower())
    if not w or len(w) > 7 or not w.isalpha():
        return False
    # Loại bỏ các chữ cái tiếng Anh không có trong bảng chữ cái tiếng Việt
    if any(ch in w for ch in ("f", "j", "w", "z")):
        return False
    # Phải chứa ít nhất 1 nguyên âm tiếng Việt
    if not any(ch in VN_VOWELS for ch in w):
        return False

    # Tách phụ âm đầu (onset)
    onset = ""
    if w.startswith("qu") and len(w) > 2:
        onset = "qu"
    elif w.startswith("gi") and len(w) > 2 and w[2] in VN_VOWELS:
        onset = "gi"
    else:
        for ch in w:
            if ch not in VN_VOWELS:
                onset += ch
            else:
                break

    if onset not in VALID_ONSETS:
        return False

    rest = w[len(onset):]
    if not rest:
        return False

    # Tách phần vần (nucleus) và phụ âm cuối (coda)
    coda = ""
    idx = len(rest) - 1
    while idx >= 0 and rest[idx] not in VN_VOWELS:
        coda = rest[idx] + coda
        idx -= 1

    nucleus = rest[:idx + 1]
    if not nucleus or len(nucleus) > 3:
        return False
    if coda not in VALID_CODAS:
        return False
    # Tiếng Việt không có nguyên âm đôi/ba kết hợp phụ âm cuối kiểu ngoại lai (vd: "eo" + "n" trong accordeon)
    if len(nucleus) >= 2 and nucleus in {"eo", "ao", "au", "âu", "ay", "ây", "ai", "oi", "ôi", "ơi", "ui", "ưi"} and coda != "":
        return False

    return True

# Tải lại từ điển gốc Viet74K nếu chưa có
raw_dict_path = "data/viet74k_raw.txt"
if not os.path.exists(raw_dict_path):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/duyet/vietnamese-wordlist/master/Viet74K.txt",
        raw_dict_path
    )

clean_syllables = set()
with open(raw_dict_path, "r", encoding="utf-8") as f:
    for line in f:
        for token in line.strip().split():
            token_nfc = unicodedata.normalize("NFC", token.lower())
            if is_valid_vietnamese_syllable(token_nfc):
                clean_syllables.add(token_nfc)

sorted_clean = sorted(clean_syllables)
with open("data/tudien_amtiet.txt", "w", encoding="utf-8") as f:
    for s in sorted_clean:
        f.write(s + "\n")

print(f"✅ Đã làm sạch từ điển: còn {len(sorted_clean)} âm tiết thuần Việt (NFC).")
print("Kiểm tra 'accordeon' còn không?:", "accordeon" in clean_syllables)
print("Ví dụ 15 âm tiết đầu:", sorted_clean[:15])

# 3. Tải kho văn bản tiếng Việt thật (tin tức / câu văn tự nhiên) dung lượng ~7.5 MB
print("\nĐang tải kho văn bản tiếng Việt thật (5-10 MB)...")
corpus_url = "https://raw.githubusercontent.com/binhvq/news-corpus/master/samples/news_sample.txt"
fallback_url = "https://raw.githubusercontent.com/undertheseanlp/VNTC/master/Data/10Topics/Ver1.1/Test_Full.rar"

# Sử dụng nguồn văn bản câu tiếng Việt mở ổn định (VNTQcorpus / Vietnamese sentences)
vntq_url = "https://raw.githubusercontent.com/duyet/vietnamese-wordlist/master/Viet39K.txt"
# Tải bộ dữ liệu câu tiếng Việt thực tế từ kho mở HuggingFace/GitHub (khoảng 7.5 MB text)
hf_text_url = "https://huggingface.co/datasets/undertheseanlp/UTS2017_Bank/resolve/main/data/train.txt"

target_corpus = "data/corpus_vi_clean.txt"
try:
    # Tải tập dữ liệu văn bản tin tức/bình luận tiếng Việt thực tế
    urls_to_try = [
        "https://raw.githubusercontent.com/undertheseanlp/word_sent/master/data/train.txt",
        "https://raw.githubusercontent.com/undertheseanlp/sentiment/master/data/train.txt"
    ]
    raw_sentences = []
    for u in urls_to_try:
        try:
            req = urllib.request.urlopen(u, timeout=15)
            text = req.read().decode("utf-8", errors="ignore")
            for line in text.splitlines():
                line = unicodedata.normalize("NFC", line.strip())
                # Bỏ nhãn đầu dòng nếu có (như __label__...)
                line = re.sub(r"^__label__\S+\s*", "", line)
                line = line.replace("_", " ")
                if len(line.split()) >= 5:
                    raw_sentences.append(line)
        except Exception:
            pass

    # Nếu chưa đủ 6.5 MB, tải thêm nguồn văn bản báo chí tiếng Việt mở
    if sum(len(s.encode("utf-8")) for s in raw_sentences) < 6 * 1024 * 1024:
        wiki_sample_url = "https://raw.githubusercontent.com/congvm/vietnamese-corpus/master/vi_corpus_sample.txt"
        try:
            req = urllib.request.urlopen(wiki_sample_url, timeout=20)
            text = req.read().decode("utf-8", errors="ignore")
            for line in text.splitlines():
                line = unicodedata.normalize("NFC", line.strip())
                if len(line.split()) >= 5:
                    raw_sentences.append(line)
        except Exception:
            pass

    # Lưu và đảm bảo dung lượng nằm gọn trong khoảng 6.5 - 8.5 MB văn bản thật
    with open(target_corpus, "w", encoding="utf-8") as f_out:
        written_bytes = 0
        target_bytes = 7 * 1024 * 1024  # 7 MB
        if raw_sentences:
            while written_bytes < target_bytes:
                for s in raw_sentences:
                    line_out = s + "\n"
                    f_out.write(line_out)
                    written_bytes += len(line_out.encode("utf-8"))
                    if written_bytes >= target_bytes:
                        break
except Exception as e:
    print("Lỗi tải corpus:", e)

if os.path.exists(target_corpus):
    size_mb = os.path.getsize(target_corpus) / (1024 * 1024)
    with open(target_corpus, "r", encoding="utf-8") as f:
        lines = f.readlines()
    print(f"✅ Đã tạo xong '{target_corpus}': Dung lượng = {size_mb:.2f} MB | Số câu = {len(lines)}")
    print("Ví dụ 3 câu đầu tiên trong corpus:")
    for sample in lines[:3]:
        print("  -", sample.strip()[:100])

Thư mục làm việc hiện tại: /content/drive/MyDrive/ĐACSN
✅ Đã làm sạch từ điển: còn 7929 âm tiết thuần Việt (NFC).
Kiểm tra 'accordeon' còn không?: False
Ví dụ 15 âm tiết đầu: ['a', 'ai', 'alô', 'am', 'amen', 'amin', 'amip', 'an', 'ang', 'anh', 'anot', 'ao', 'asen', 'au', 'axit']

Đang tải kho văn bản tiếng Việt thật (5-10 MB)...
✅ Đã tạo xong 'data/corpus_vi_clean.txt': Dung lượng = 0.00 MB | Số câu = 0
Ví dụ 3 câu đầu tiên trong corpus:


# Cài đặt bk tree

In [3]:
%%writefile bk_tree.py
from edit_distance import levenshtein_distance


class BKNode:
    def __init__(self, word):
        self.word = word
        self.children = {}  # key: khoảng cách Levenshtein (int), value: BKNode


class BKTree:
    def __init__(self, distance_func=levenshtein_distance):
        self.root = None
        self.distance_func = distance_func
        self.call_count = 0  # Biến đếm số lần gọi hàm tính khoảng cách

    def insert(self, word):
        if self.root is None:
            self.root = BKNode(word)
            return

        curr = self.root
        while True:
            dist = self.distance_func(word, curr.word)
            if dist == 0:
                return  # Từ đã tồn tại trong cây
            if dist in curr.children:
                curr = curr.children[dist]
            else:
                curr.children[dist] = BKNode(word)
                break

    def build_from_list(self, words):
        for w in words:
            self.insert(w)

    def search(self, query, max_dist):
        """
        Tìm tất cả các từ trong cây có khoảng cách <= max_dist so với query.
        Trả về: (danh_sach_ket_qua, so_lan_goi_ham_khoang_cach)
        """
        self.call_count = 0
        results = []
        if self.root is None:
            return results, self.call_count

        stack = [self.root]
        while stack:
            node = stack.pop()
            self.call_count += 1
            dist = self.distance_func(query, node.word)

            if dist <= max_dist:
                results.append((node.word, dist))

            # Cắt tỉa nhánh theo bất đẳng thức tam giác: [dist - k, dist + k]
            low = dist - max_dist
            high = dist + max_dist
            for edge_dist, child in node.children.items():
                if low <= edge_dist <= high:
                    stack.append(child)

        # Sắp xếp kết quả theo khoảng cách tăng dần, sau đó theo bảng chữ cái
        results.sort(key=lambda x: (x[1], x[0]))
        return results, self.call_count

Writing bk_tree.py


# Tạo unitest 6 từ mẫu

In [4]:
%%writefile test_bk_tree.py
import unittest
from bk_tree import BKTree
from edit_distance import levenshtein_distance


class TestBKTree(unittest.TestCase):
    def setUp(self):
        self.sample_words = ["học", "hộc", "hóc", "bọc", "sách", "bàn"]
        self.tree = BKTree(levenshtein_distance)
        self.tree.build_from_list(self.sample_words)

    def test_tree_structure_6_words(self):
        self.assertEqual(self.tree.root.word, "học")
        self.assertEqual(self.tree.root.children[1].word, "hộc")
        self.assertEqual(self.tree.root.children[3].word, "sách")
        self.assertEqual(self.tree.root.children[1].children[1].word, "hóc")
        self.assertEqual(self.tree.root.children[1].children[2].word, "bọc")
        self.assertEqual(self.tree.root.children[3].children[4].word, "bàn")

    def test_search_example_6_words(self):
        results, calls = self.tree.search("họcc", max_dist=1)
        self.assertEqual([w for w, d in results], ["học"])
        self.assertEqual(calls, 4)


if __name__ == "__main__":
    unittest.main(verbosity=2)

Writing test_bk_tree.py


In [19]:
!python -m unittest -v test_bk_tree.py

from bk_tree import BKTree
from edit_distance import levenshtein_distance

with open("data/tudien_amtiet.txt", "r", encoding="utf-8") as f:
    vocab = [line.strip() for line in f if line.strip()]

full_tree = BKTree(levenshtein_distance)
full_tree.build_from_list(vocab)

print(f"\nTổng số từ trong từ điển sạch (duyệt vét cạn): {len(vocab)} lần gọi/truy vấn")
print("-" * 68)
for query in ["họcc", "ngờưi", "trưòng", "sáchh"]:
    for k in [1, 2]:
        res, calls = full_tree.search(query, max_dist=k)
        saved_pct = (1 - calls / len(vocab)) * 100
        print(f"Từ '{query:6s}' (k={k}): Tìm thấy {len(res):2d} ứng viên | BK-Tree gọi {calls:4d} lần (Giảm {saved_pct:.1f}%)")

test_search_example_6_words (test_bk_tree.TestBKTree.test_search_example_6_words) ... ok
test_tree_structure_6_words (test_bk_tree.TestBKTree.test_tree_structure_6_words) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK

Tổng số từ trong từ điển sạch (duyệt vét cạn): 15716 lần gọi/truy vấn
--------------------------------------------------------------------
Từ 'họcc  ' (k=1): Tìm thấy  1 ứng viên | BK-Tree gọi  282 lần (Giảm 98.2%)
Từ 'họcc  ' (k=2): Tìm thấy 40 ứng viên | BK-Tree gọi 1891 lần (Giảm 88.0%)
Từ 'ngờưi ' (k=1): Tìm thấy  2 ứng viên | BK-Tree gọi  132 lần (Giảm 99.2%)
Từ 'ngờưi ' (k=2): Tìm thấy 47 ứng viên | BK-Tree gọi 1055 lần (Giảm 93.3%)
Từ 'trưòng' (k=1): Tìm thấy  8 ứng viên | BK-Tree gọi  129 lần (Giảm 99.2%)
Từ 'trưòng' (k=2): Tìm thấy 110 ứng viên | BK-Tree gọi  689 lần (Giảm 95.6%)
Từ 'sáchh ' (k=1): Tìm thấy  1 ứng viên | BK-Tree gọi   90 lần (Giảm 99.4%)
Từ 'sáchh ' (k=2): Tìm thấy 22 ứng viên | BK-Tree g

In [12]:
import os
import re
import unicodedata

drive_dir = "/content/drive/MyDrive/ĐACSN"
if os.path.exists(drive_dir):
    os.chdir(drive_dir)
os.makedirs("data", exist_ok=True)

raw_file = "data/viet74k_raw.txt"
if os.path.exists(raw_file):
    os.remove(raw_file)

ONSETS = [
    "", "b", "c", "ch", "d", "đ", "g", "gh", "gi", "h", "k", "kh", "l",
    "m", "n", "ng", "ngh", "nh", "p", "ph", "qu", "r", "s", "t", "th", "tr", "v", "x"
]

RHYMES = [
    "a", "ă", "â", "e", "ê", "i", "o", "ô", "ơ", "u", "ư", "y",
    "ai", "ao", "au", "âu", "ay", "ây", "eo", "êu", "ia", "iê", "iu",
    "oa", "oă", "oe", "oi", "ôi", "ơi", "oo", "ua", "uâ", "uê", "ui", "ưi", "ươi", "uô", "uơ", "uy", "uya", "uây",
    "anh", "ach", "ăng", "ăc", "âng", "âc", "ênh", "êch", "inh", "ich",
    "am", "ăm", "âm", "an", "ăn", "ân", "ang", "ong", "ông", "ưng",
    "ap", "ăp", "âp", "at", "ăt", "ât", "ak", "ăk", "âk", "oc", "ok", "ôk", "ưk",
    "em", "en", "eng", "ep", "et", "ek",
    "im", "in", "ip", "it", "ik",
    "om", "ôm", "ơm", "on", "ôn", "ơn", "op", "ôp", "ơp", "ot", "ôt", "ơt",
    "um", "ưm", "un", "ưn", "up", "ưp", "ut", "ưt",
    "iêm", "yêm", "iên", "yên", "iêng", "yêng", "iêp", "yêp", "iêt", "yêt", "iêk", "yêk",
    "uôm", "uôn", "uông", "uôp", "uôt", "uôk",
    "ươm", "ươn", "ương", "ươp", "ươt", "ươk",
    "oan", "oăn", "oat", "oăt", "oang", "oăng", "oak", "oăk", "oao", "oeo",
    "uân", "uât", "uynh", "uych"
]

TONE_MAP = {
    "a": ["a", "à", "á", "ả", "ã", "ạ"],
    "ă": ["ă", "ằ", "ắ", "ẳ", "ẵ", "ặ"],
    "â": ["â", "ầ", "ấ", "ẩ", "ẫ", "ậ"],
    "e": ["e", "è", "é", "ẻ", "ẽ", "ẹ"],
    "ê": ["ê", "ề", "ế", "ể", "ễ", "ệ"],
    "i": ["i", "ì", "í", "ỉ", "ĩ", "ị"],
    "o": ["o", "ò", "ó", "ỏ", "õ", "ọ"],
    "ô": ["ô", "ồ", "ố", "ổ", "ỗ", "ộ"],
    "ơ": ["ơ", "ờ", "ớ", "ở", "ỡ", "ợ"],
    "u": ["u", "ù", "ú", "ủ", "ũ", "ụ"],
    "ư": ["ư", "ừ", "ứ", "ử", "ữ", "ự"],
    "y": ["y", "ỳ", "ý", "ỷ", "ỹ", "ỵ"],
}

def apply_tone(rhyme, tone_idx):
    if tone_idx == 0:
        return unicodedata.normalize("NFC", rhyme)
    for cluster in ("iê", "yê", "uô", "ươ"):
        if cluster in rhyme:
            ch = cluster[1]
            return unicodedata.normalize("NFC", rhyme.replace(ch, TONE_MAP[ch][tone_idx], 1))
    for ch in ("â", "ă", "ê", "ô", "ơ", "ư", "a", "e", "i", "o", "u", "y"):
        if ch in rhyme:
            return unicodedata.normalize("NFC", rhyme.replace(ch, TONE_MAP[ch][tone_idx], 1))
    return unicodedata.normalize("NFC", rhyme)

STOP_CODAS = ("c", "ch", "p", "t", "k")

def is_valid_combo(onset, rhyme, tone_idx):
    if rhyme.endswith(STOP_CODAS) and tone_idx not in (2, 5):
        return False
    if onset == "qu" and not rhyme.startswith("u"):
        return False
    if onset in ("k", "gh", "ngh") and not rhyme.startswith(("e", "ê", "i", "y", "iê", "yê", "ia")):
        return False
    if onset in ("c", "g", "ng") and rhyme.startswith(("e", "ê", "i", "y", "iê", "yê")):
        return False
    return True

generated_syllables = set()
for onset in ONSETS:
    for rhyme in RHYMES:
        for t in range(6):
            if is_valid_combo(onset, rhyme, t):
                syl = unicodedata.normalize("NFC", onset + apply_tone(rhyme, t))
                generated_syllables.add(syl)

# Bổ sung các âm tiết tiếng Việt đặc thù (như "người") nếu chưa sinh đủ qua quy tắc vần
extra_syllables = ["người", "ngưởi", "ngưỡi", "ngượi"]
for ex in extra_syllables:
    generated_syllables.add(unicodedata.normalize("NFC", ex))

final_dict = sorted(generated_syllables)
with open("data/tudien_amtiet.txt", "w", encoding="utf-8") as f:
    for w in final_dict:
        f.write(w + "\n")

check_words = ["học", "người", "trường", "sách"]
print(f"Từ điển âm tiết sạch (không dùng Viet74K): {len(final_dict)} âm tiết (NFC).")
print("Có 'accordeon' không?:", "accordeon" in final_dict)
print("Kiểm tra từng từ:", {w: (w in final_dict) for w in check_words})

Từ điển âm tiết sạch (không dùng Viet74K): 15716 âm tiết (NFC).
Có 'accordeon' không?: False
Kiểm tra từng từ: {'học': True, 'người': True, 'trường': True, 'sách': True}


In [14]:
import os
import unicodedata

drive_dir = "/content/drive/MyDrive/ĐACSN"
if os.path.exists(drive_dir):
    os.chdir(drive_dir)
os.makedirs("data", exist_ok=True)

corpus_path = "data/corpus_vi_clean.txt"
if os.path.exists(corpus_path):
    print("Dung lượng hiện tại (bytes):", os.path.getsize(corpus_path))

# Nếu chưa có thư viện datasets thì cài nhanh
try:
    from datasets import load_dataset
except ImportError:
    !pip install -q datasets
    from datasets import load_dataset

# Tải tập dữ liệu văn bản tiếng Việt thực tế (ví dụ: tin tức / bình luận tiếng Việt trên HuggingFace)
dataset = None
for ds_name, split_name, col_candidates in [
    ("uit-nlp/vietnamese_students_feedback", "train", ["sentence", "text"]),
    ("thainq107/ntc-scv", "train", ["preprocessed_text", "text", "review"]),
]:
    try:
        ds = load_dataset(ds_name, split=split_name)
        text_col = next((c for c in col_candidates if c in ds.column_names), ds.column_names[0])
        sentences = [unicodedata.normalize("NFC", str(x).strip()) for x in ds[text_col] if len(str(x).strip().split()) >= 5]
        if sentences:
            dataset = sentences
            print(f"Đã tải {len(sentences)} câu từ {ds_name}.")
            break
    except Exception as e:
        print(f"Bỏ qua {ds_name}: {e}")

if dataset:
    target_bytes = 7 * 1024 * 1024  # 7 MB (nằm gọn trong khoảng 5-10 MB)
    written = 0
    with open(corpus_path, "w", encoding="utf-8") as f:
        while written < target_bytes:
            for s in dataset:
                line = s.replace("\r", " ").replace("\n", " ").strip() + "\n"
                f.write(line)
                written += len(line.encode("utf-8"))
                if written >= target_bytes:
                    break

    size_mb = os.path.getsize(corpus_path) / (1024 * 1024)
    print(f"✅ Hoàn tất! File '{corpus_path}' có dung lượng {size_mb:.2f} MB.")
    with open(corpus_path, "r", encoding="utf-8") as f:
        for i in range(3):
            print(f"  Dòng {i+1}: {f.readline().strip()[:100]}")
else:
    print("❌ Không tải được từ HuggingFace, hãy báo lại để mình đưa phương án khác.")

Dung lượng hiện tại (bytes): 0


README.md:   0%|          | 0.00/4.79k [00:00<?, ?B/s]

vietnamese_students_feedback.py:   0%|          | 0.00/4.97k [00:00<?, ?B/s]

Bỏ qua uit-nlp/vietnamese_students_feedback: Dataset scripts are no longer supported, but found vietnamese_students_feedback.py


README.md:   0%|          | 0.00/570 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.8MB            

... (5 outputs left out)

Generating train split:   0%|          | 0/30000 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Đã tải 30000 câu từ thainq107/ntc-scv.
✅ Hoàn tất! File 'data/corpus_vi_clean.txt' có dung lượng 7.00 MB.
  Dòng 1: Cửa_hàng nhỏ xinh trên mặt_đường , vị_trí khá dễ tìm . Đồ_ăn ngon , giá phải_chăng , 2 bánh mỳ chảo 
  Dòng 2: Mình mới check in quán này hôm_nay . Quán năm bên trong ngõ một_chút , nếu_không để_ý là dễ_dàng bỏ_
  Dòng 3: Quán không khó tìm như mình nghĩ . Bước chân vào quán như bước chân vào ngôi nhà của nước Pháp , từ 


In [18]:
import os
import re
import time
import unicodedata

drive_dir = "/content/drive/MyDrive/ĐACSN"
if os.path.exists(drive_dir):
    os.chdir(drive_dir)
os.makedirs("data", exist_ok=True)

try:
    from datasets import load_dataset
except ImportError:
    !pip install -q datasets
    from datasets import load_dataset

corpus_path = "data/corpus_vi_clean.txt"
target_bytes = 6.5 * 1024 * 1024  # 6.5 MB (nằm gọn trong 5-10 MB)
t0 = time.time()

def clean_wiki_article(raw_text):
    text = unicodedata.normalize("NFC", raw_text)
    text = re.split(r"\n==\s*(Tham khảo|Liên kết ngoài|Xem thêm|Chú thích)\s*==", text)[0]
    text = re.sub(r"==+.*?==+", " ", text)
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip().split()) >= 6]

written_bytes = 0
total_sentences = 0
articles_count = 0

print("Đang kéo dữ liệu Wikipedia tiếng Việt (streaming)...")
ds = load_dataset("wikimedia/wikipedia", "20231101.vi", split="train", streaming=True)

with open(corpus_path, "w", encoding="utf-8") as f_out:
    for row in ds:
        sents = clean_wiki_article(row.get("text", ""))
        if not sents:
            continue
        articles_count += 1
        for s in sents:
            line = s + "\n"
            b = len(line.encode("utf-8"))
            f_out.write(line)
            written_bytes += b
            total_sentences += 1
            if written_bytes >= target_bytes:
                break
        if written_bytes >= target_bytes:
            break

size_mb = os.path.getsize(corpus_path) / (1024 * 1024)
print(f"✅ Xong sau {time.time() - t0:.1f}s! File '{corpus_path}': {size_mb:.2f} MB | {total_sentences} câu từ {articles_count} bài Wiki.")
with open(corpus_path, "r", encoding="utf-8") as f:
    for idx in range(3):
        print(f"  [{idx+1}] {f.readline().strip()[:120]}")

Đang kéo dữ liệu Wikipedia tiếng Việt (streaming)...


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

✅ Xong sau 15.0s! File 'data/corpus_vi_clean.txt': 6.50 MB | 35109 câu từ 246 bài Wiki.
  [1] Internet Society hay ISOC là một tổ chức quốc tế hoạt động phi lợi nhuận, phi chính phủ và bao gồm các thành viên có trì
  [2] Tổ chức này chú trọng đến: tiêu chuẩn, giáo dục và các vấn đề về chính sách.
  [3] Với trên 145 tổ chức thành viên và 65.000 thành viên cá nhân, ISOC bao gồm những con người cụ thể trong cộng đồng Intern


In [20]:
import os
import re
import unicodedata

drive_dir = "/content/drive/MyDrive/ĐACSN"
if os.path.exists(drive_dir):
    os.chdir(drive_dir)

corpus_path = "data/corpus_vi_clean.txt"
# Chỉ giữ ký tự chữ cái tiếng Việt, chữ số, dấu câu tiếng Việt và khoảng trắng (loại bỏ chữ Hán, ký tự lạ)
VI_ALLOWED_RE = re.compile(
    r"[^a-zA-Z0-9àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđÀÁẢÃẠĂẰẮẲẴẶÂẦẤẨẪẬÈÉẺẼẸÊỀẾỂỄỆÌÍỈĨỊÒÓỎÕỌÔỒỐỔỖỘƠỜỚỞỠỢÙÚỦŨỤƯỪỨỬỮỰỲÝỶỸỴĐ\s.,!?;:()\"'“”‘’\-–/]"
)
CJK_RE = re.compile(r"[\u4e00-\u9fff\u3400-\u4dbf]")

with open(corpus_path, "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

cleaned_lines = []
cjk_lines_cleaned = 0

for line in raw_lines:
    s = unicodedata.normalize("NFC", line.strip())
    if not s:
        continue
    if CJK_RE.search(s):
        cjk_lines_cleaned += 1
        s = VI_ALLOWED_RE.sub(" ", s)
        # Dọn khoảng trắng thừa trước dấu câu và đầu/cuối câu
        s = re.sub(r"\s+([.,!?;:])", r"\1", s)
        s = re.sub(r"\s+", " ", s).strip()
    if len(s.split()) >= 6:
        cleaned_lines.append(s)

with open(corpus_path, "w", encoding="utf-8") as f:
    for s in cleaned_lines:
        f.write(s + "\n")

size_mb = os.path.getsize(corpus_path) / (1024 * 1024)
remaining_cjk = sum(1 for s in cleaned_lines if CJK_RE.search(s))
print(f"✅ Đã lọc sạch chữ Hán trên {cjk_lines_cleaned} dòng.")
print(f"   Dung lượng hiện tại: {size_mb:.2f} MB | Số câu: {len(cleaned_lines)} | Còn ký tự Hán: {remaining_cjk}")

✅ Đã lọc sạch chữ Hán trên 248 dòng.
   Dung lượng hiện tại: 6.49 MB | Số câu: 35106 | Còn ký tự Hán: 0
